# SE1_data_access


In [1]:
#Modify cell
#load dependencies
from pathlib import Path
import requests
import pandas as pd
import numpy as np
import geopandas as gpd
import rasterio as rio
import shutil
import zipfile
import os
import io
from shapely.geometry import box
import time
import urllib.request
from urllib.parse import urlparse

<span style='color:green'>keep</span> 

All downloaded data are stored **outside the repository** to:

- keep the repo lightweight. Note: never store files more than 50 MB in your research repository, this will violate GitHub use policies.
- prevent large file versioning.
- maintain reproducibility.
- separate raw vs processed data.

The location of raw data is determined automatically using the project name.

In [2]:
# Initiate rawdata download directory

# Automaticallly generated cell, create download folder for "raw data" outside of your project directory
# Current working directory = project/notebooks
notebook_dir = Path.cwd()

# Project root (one level up from notebooks)
project_root = notebook_dir.parent

# Parent directory OUTSIDE the repository
external_base = project_root.parent

# Create raw data directory outside repo
raw_data_dir = external_base / "{{ cookiecutter.project_slug }}_rawdata"

raw_data_dir.mkdir(parents=True, exist_ok=True)

print("External raw data directory:")
print(raw_data_dir.resolve())


External raw data directory:
/home/jovyan/{{ cookiecutter.project_slug }}_rawdata


<span style='color:goldenrod'>modify before publishing</span> 
## Data description
Describe your data sources here. Reference the table provided on your README.

## How to run this notebook
Provide a detailed description of inputs, input directories, outputs, and output directories.

## Storage requirements
Total required storage: what is the total volume of data that will be downloaded with this notebook?
Maximum file size: what is the largest individual file size that will be downloaded with this notebook?

### Make sure you have sufficient storage available to download all data required

In [3]:
!df -k ..
# the table below prints out how much storate is available in the folder containing your raw_data download directory

Filesystem     1K-blocks    Used Available Use% Mounted on
/dev/sdc       514937088 1728260 513192444   1% /home/jovyan


<span style='color:red'>delete before publishing</span> 
# Methods to automate data access 
Accessing data automatically using URLs (uniform resource locators, also known as web addresses) is fundamental to FAIR research (Findable, Accessible, Interoperable, Reproducible), in that it allows for others to access your data without creating redundant copies of files. We are particularly interested in using persistent identfiers (PIDs), which are URLs that do not change over time. One type of PID that we are very familiar with in research is a DOI (Digital Object Identifier). These are assigned to our publications, for example. 

There are three options to specify download URLs in this code:
1) List stable URLs in a "datalinks.txt" file in your "inputs" folder, and then iterate over these to download them. This option is ideal for smaller files that do not require user authentication to access.
2) Generate list of stable URLs using database intuition: this is ideal for ftp and http servers, where databasing syntax is easily inferred from the file URLs, without need for an API (advanced programming interface) to modify the search. Here, we can use regular expressions to guess the URLs for data representing different locations, points in time, or variables, or customize URLs to include details like our own username and password.
3) Accessing data though an API (Advanced Programming Interface): APIs allow us to use code to interact with database search parameters, find relavant data, and access download links for that relevant data. APIs help with searching complex, heterogenous geodatabases. For very large datasets (reanalysis data, satellite remote sensing data, etc), certain APIs allow us to subset data (say for a specific location) prior to downloading, reducing storage requirements. APIs often require environmnetal configuration for access control. If so, **access configuration details must be specified in README**

## Handling generated or simulated data
For experiments relying on synthetic data, please include synthetic data generation script in this notebook. Be sure to use features like set seed to ensure consistency with random number generation. Save all outputs to rawdata folder.

<span style='color:red'>delete before publishing</span> 
# Define download file helper functions
This function will download files from URLs to the raw data folder created in the parent folder of the current repository. 


In [4]:
def get_filename_from_url(url):
    path = urlparse(url).path
    name = os.path.basename(path)
    return name if name else "downloaded_file"


def download_file(url, output_dir):
    filename = get_filename_from_url(url)
    filepath = os.path.join(output_dir, filename)

    if os.path.exists(filepath):
        print(f"File already exists, skipping: {filename}")
        return

    try:
        print(f"Downloading: {filename}")
        with requests.get(url, stream=True, timeout=TIMEOUT) as r:
            r.raise_for_status()
            with open(filepath, "wb") as f:
                for chunk in r.iter_content(chunk_size=8192):
                    if chunk:
                        f.write(chunk)
        print(f"✅ Saved: {filepath}")

    except Exception as e:
        print(f" Failed: {url}")
        print(f" Error: {e}")


<span style='color:red'>delete before publishing</span> 

# Example option 1: Downloading from a list of URLs in inputs/datalinks.txt
To use this option, copy and paste download urls of relevant datasets into the text folder located at inputs/datalinks.txt.

In [5]:
# ===== USER SETTINGS =====
LINKS_FILE = "inputs/datalinks.txt"
OUTPUT_DIR =  raw_data_dir.resolve()
DELAY_SECONDS = 2      # Set to 0 if no delay needed, check your data provider's API access instructions to see what delay requirements exist
TIMEOUT = 60           # Seconds before request timeout, check your data provider's API access instructions to change as needed.
# =========================

In [6]:
from pathlib import Path
print(Path.cwd())

/home/jovyan/WaterDig/Yanni


In [7]:
from pathlib import Path

inputs_path = Path("inputs")
inputs_path.mkdir(exist_ok=True)

file_path = inputs_path / "datalinks.txt"

with open(file_path, "w") as f:
    f.write("https://downloads.psl.noaa.gov/Datasets/ncep.reanalysis.dailyavgs/surface/air.sig995.2010.nc\n")

print("Created:", file_path.resolve())

Created: /home/jovyan/WaterDig/Yanni/inputs/datalinks.txt


In [8]:
with open(LINKS_FILE, "r") as f:
    urls = [line.strip() for line in f if line.strip()]

print(f"Found {len(urls)} links.\n")

for i, url in enumerate(urls, 1):
    print(f"[{i}/{len(urls)}]")
    download_file(url, OUTPUT_DIR)

    if DELAY_SECONDS > 0 and i < len(urls):
        print(f"⏳ Waiting {DELAY_SECONDS} seconds...\n")
        time.sleep(DELAY_SECONDS)

print("\n🎉 Done.")


Found 1 links.

[1/1]
File already exists, skipping: air.sig995.2010.nc

🎉 Done.


<span style='color:red'>delete before publishing</span> 
# Option 2: using regular expressions to define URLs

## Decoding the file structure of online geodatabases
Just like we can use code to find and access files on our local machine, we can use code to find and access files on public geodatabases. Since these geodatabases are version controlled, providing code that links to the online files helps prevent us from making redundant copies of data on the internet. Programatically accessing public geodatabases requires that we understand how the database itself has been organized.

 - Click on the following link to the National Oceanic and Atmospheric Association databse website: https://psl.noaa.gov/data/gridded/

 - Navigate to the "NCEP/NCAR Reanalysis dataset"
 - Of the seven sections they've divided data into, click on "Surface"
 - Under "Air Temperature: Daily", click "See list"
 - Under "Surface", click "See list"

### TASK: Right click on the first link in the list, and select "copy link". Paste that link address below:
https://downloads.psl.noaa.gov/Datasets/ncep.reanalysis.dailyavgs/surface/air.sig995.1948.nc

There is anatomy to this link.
* https://downloads.psl.noaa.gov/Datasets: this is the stable location of many different types surface level (psl) meteorological data stored by NOAA.gov
* ncep.reanalsys.dailavgs : this gives us information about a specific dataset, here we are accessing the NCAR NCEP reanalysis data, daily averages
* air.sig995. : this tells us we are getting air tempearture at the near surface (indicated here by geopotential height of 995mBar pressure level
* 1948 : this is a year
* .nc : this is our file extension

In the link above, we can break the filepath down into substrings, using basic text commands:

In [9]:
# ===== OPTION 2 USER SETTINGS =====
OUTPUT_DIR =  raw_data_dir.resolve()
DELAY_SECONDS = 2      # Set to 0 if no delay needed, check your data provider's API access instructions to see what delay requirements exist
TIMEOUT = 60           # Seconds before request timeout, check your data provider's API access instructions to change as needed.
# =========================

In [10]:
http_dir = "https://downloads.psl.noaa.gov/Datasets/"
dataset = "ncep.reanalysis.dailyavgs"
lev_type = "surface"
variable = "air.sig995."
year = "2010"
file_type = ".nc"
filepaths = http_dir + dataset + "/" + lev_type + "/" + variable + year + file_type
print(filepaths)

https://downloads.psl.noaa.gov/Datasets/ncep.reanalysis.dailyavgs/surface/air.sig995.2010.nc



**Notice here that I change the "year" to 2010. If we click on that link, what happens?**

In [11]:
url = filepaths
filename = str(raw_data_dir) + variable + year+ file_type
urllib.request.urlretrieve(url, filename)
print(url, filename)

https://downloads.psl.noaa.gov/Datasets/ncep.reanalysis.dailyavgs/surface/air.sig995.2010.nc /home/jovyan/{{ cookiecutter.project_slug }}_rawdataair.sig995.2010.nc


In [12]:
!ls
#Do you see the file listed in your current working directory?

 inputs		       'SE1_data_access_Yanni Yang.ipynb'       yanni0502_2.txt
 Oulanka_3D.png         SE2_data_processing.ipynb	        yanni0502.txt
 README_note-Copy1.md  'SE2_data_processing_Yanni Yang.ipynb'   yanni2901.txt
 README_Yanni.md        yanni0502_2


We can use regular expressions or predictable expressions in the URL structure to automatically create a list of URLs. For example, I can use the "range" function to create a list of years from 1965 -1985

In [13]:
year =pd.Series(list(range(1965,1975)))
year = year.apply(str)
filepaths= http_dir + dataset + "/" + lev_type + "/" + variable + year + file_type
print(filepaths)

0    https://downloads.psl.noaa.gov/Datasets/ncep.r...
1    https://downloads.psl.noaa.gov/Datasets/ncep.r...
2    https://downloads.psl.noaa.gov/Datasets/ncep.r...
3    https://downloads.psl.noaa.gov/Datasets/ncep.r...
4    https://downloads.psl.noaa.gov/Datasets/ncep.r...
5    https://downloads.psl.noaa.gov/Datasets/ncep.r...
6    https://downloads.psl.noaa.gov/Datasets/ncep.r...
7    https://downloads.psl.noaa.gov/Datasets/ncep.r...
8    https://downloads.psl.noaa.gov/Datasets/ncep.r...
9    https://downloads.psl.noaa.gov/Datasets/ncep.r...
dtype: str


In [14]:
print(f"Found {len(filepaths)} links.\n")
for i in range(len(filepaths)):
    
    print(f"[{i}/{len(filepaths)}]")
    url= filepaths[i]
    download_file(url, OUTPUT_DIR)
    if DELAY_SECONDS > 0 and i < len(filepaths):
        print(f"⏳ Waiting {DELAY_SECONDS} seconds...\n")
        time.sleep(DELAY_SECONDS)

print("\n Done.")

Found 10 links.

[0/10]
File already exists, skipping: air.sig995.1965.nc
⏳ Waiting 2 seconds...

[1/10]
File already exists, skipping: air.sig995.1966.nc
⏳ Waiting 2 seconds...

[2/10]
File already exists, skipping: air.sig995.1967.nc
⏳ Waiting 2 seconds...

[3/10]
File already exists, skipping: air.sig995.1968.nc
⏳ Waiting 2 seconds...

[4/10]
File already exists, skipping: air.sig995.1969.nc
⏳ Waiting 2 seconds...

[5/10]
File already exists, skipping: air.sig995.1970.nc
⏳ Waiting 2 seconds...

[6/10]
File already exists, skipping: air.sig995.1971.nc
⏳ Waiting 2 seconds...

[7/10]
File already exists, skipping: air.sig995.1972.nc
⏳ Waiting 2 seconds...

[8/10]
File already exists, skipping: air.sig995.1973.nc
⏳ Waiting 2 seconds...

[9/10]
File already exists, skipping: air.sig995.1974.nc
⏳ Waiting 2 seconds...


 Done.


<span style='color:red'>delete before publishing</span> 
# Option 3: generate list of filepaths using API for an online geodatabase
#TODO: Update with AquaINFRA


There are dozens of rich online geodatabases for governmental, public, and research data available. Most of this geodatabases will provide both a graphical user interface (GUI) and advanced programming interface (API) to search, select, and download their data holdings. The code below was generated using the Copernicus Climate Data Store (CDS) web-based GUI. This graphical user interface returns a scripted python API call that I can copy into my notebook to download my data. To generate this code, I went to the following website:

(https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-monthly-means?tab=download)

And filled out the form to specify my search parameters. I then went to the bottom of the page, and under "Corresponding API Code", seleceted "Show API Request Code"

Please note that I must download the cdsapi python package to execute this request, and indicate my Copernicus login credentials by modifying the $HOME/.cdsapirc file created by this python library for this to work.
[How to configure the Copernicus Climate Data Store API on your machine](https://cds.climate.copernicus.eu/how-to-api)

Learn more about geodatabase authentication protocols on the "Data access authentication" notebook in the how-to page on https://github.com/DigitalWaters-fi.

In [15]:
import cdsapi

dataset = "reanalysis-era5-single-levels-monthly-means"
request = {
    "product_type": ["monthly_averaged_reanalysis"],
    "variable": [
        "2m_temperature",
        "total_precipitation",
        "clear_sky_direct_solar_radiation_at_surface",
        "instantaneous_surface_sensible_heat_flux",
        "surface_latent_heat_flux",
        "runoff"
    ],
    "year": [
        "2000", "2001", "2002",
        "2003", "2004", "2005",
        "2006", "2007", "2008",
        "2009", "2010", "2011",
        "2012", "2013", "2014",
        "2015", "2016", "2017",
        "2018", "2019", "2020"
    ],
    "month": [
        "01", "02", "03",
        "04", "05", "06",
        "07", "08", "09",
        "10", "11", "12"
    ],
    "time": ["00:00"],
    "data_format": "grib",
    "download_format": "unarchived",
    "area": [72, 19, 58, 32]
}

client = cdsapi.Client()
client.retrieve(dataset, request).download()


ModuleNotFoundError: No module named 'cdsapi'

## Need help?

Search, ask, and answer data access questions at https://github.com/orgs/DigitalWaters-fi/discussions

Tag #dataaccess